## Importaciones

In [ ]:
from funciones_BBDD import *
import pandas as pd

# DRUGBANK 1
DataFrame general

In [ ]:
xml_file = "./drugbank_all_full_database.xml/full_database.xml"
output_file = "enzimas.csv"

csv_enzimas(output_file, xml_file)

In [ ]:
df_drugbank = pd.read_csv ('enzimas.csv')

print("Principios activos:", len(df_drugbank["drugbank_id"].unique()))
df_drugbank.head()

### Descripcion
1. **drugbank_id** Identificador único de DrugBank para el principio activo
2. **drug_name** Nombre del principio activo (genérico)
3. **synonimous** Nombres alternativos para el principio activo (comerciales/ codigos de investigacion/ en otros idiomas)
4. **type** Categoria de la proteína. Rol biologico de la proteína con la que interactúa el principio activo
    - Target. Proteina diana, donde el farmaco ejerce su efecto principal
    - Enzyme. proteinas que metabolizan el fármaco
5. **Gene_name** Símbolo del gen que codifica esa proteína
6. **uniprot_id** Codigo de acceso para UniProt
7. **Action** Accion del principio activo a la proteína
  


In [ ]:
df_drugbank['type'].unique()

In [ ]:
df_drugbank['action'].unique()

In [ ]:
df_drugbank.isna().any()

In [ ]:
#No hay elementos de accion nulos solo con las enzimas
enz = df_drugbank[df_drugbank["type"]=="enzyme"]
enz.isna().any()

# DRUGBANK 2
DF con los codigos ATC (se añadió mas tarde cuando se vio que era importante para establecer la prioridad)

In [ ]:
xml_file = "./drugbank_all_full_database.xml/full_database.xml"
output_file = "ATC_DB.csv"

csv_ATC_db(xml_file, output_file)

In [ ]:
df_ATC = pd.read_csv ('ATC_DB.csv')

print("Principios activos:", len(df_ATC["drugbank_id"].unique()))
df_ATC.head()

### Descripcion
1. **drugbank_id**
2. **drug_name**
3. **atc_codes**

# SIDDER (efectos adversos)

In [ ]:
path_se="meddra_all_se.tsv",
path_freq="meddra_freq.tsv",
path_names="drug_names.tsv",
output_csv="sidder_cambio.csv"

#Filtrados por prefered term (PT)
csv_sidder(path_se, path_freq, path_names, output_csv)

In [ ]:
df_sidder = pd.read_csv ('sidder_cambio.csv')

print("Diferentes id:", len(df_drugbank["flat"].unique()))
df_drugbank.head()

### Descripcion
1. **flat**
2. **drug_name**
3. **side_effect_se**
4. **freq_mean_y**

In [ ]:
df_drugbank.isna().any()

# Establecemos prioridad de enzimas
Para ello se usa la base de datos de CIMA

In [ ]:
#Con este comando obetenemos: medicamentos.csv
main()


In [ ]:
#Abrimos el csv de CIMA
med = pd.read_csv("medicamentos.csv")
med.head()

### Descripcion
1. **nombre_medicamento**
2. **numero_resgistro**
3. **principios_activos**
4. **codigo_atc**
5. **forma_farmaceutica**
6. **situacion_registro**

In [ ]:
# Elimino nulos (los estoy eliminando todos, deberia eliminar solo los que sean nulos en el principio activo?) NO HAY CAMBIO
med = med.dropna(subset=["principios_activos"])
# Me quedo solo con las que tengan un principio activo
uno = med[~med["principios_activos"].str.contains("/", na=False)]
#Me quedo con un solo medicamento  para ver la prioridad de enzima de metabolizacion que tiene un principio activo
prioridad = uno.drop_duplicates(subset=["principios_activos"], keep = 'first')

print(f"Los medicamentos existentes en CIMA son: {len(med)}")
print(f"Los medicamentos con tan solo un principio activo resgistrados en CIMA son: {len(uno)}")
print(f"Solo un medicamento con un principio activo: {len(prioridad)}")

In [ ]:
# Lo intentamos con el nombre del principio activo: no funciono
# Lo intenamos con los sinonimos tambien: tampoco se pudo
# Se optó por los codigos ATC para identificar los principios activos entre bases de datos diferentes

In [ ]:
# Diccionario que cuenta cuantas encimas tengo de cada y cuales son el top 5 con las que me quedare
diccionario_enzimas = (enzimas[enzimas["type"] == "enzyme"]["gene_name"].value_counts().to_dict())

print(diccionario_enzimas)

In [ ]:
#Por tanto escogemos las que se metabolizan por cualquiera de las siguientes enzimas ["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"] 
#que son las top 5 y CYP3A5 porque va con CYP3A4 (explicar mejor)

#Me quedo solo con las enzimas
enzimas = df_drugbank[df_drugbank["type"]=="enzyme"]
#Las que queremos (pongo tambien 3A5)
top_5 = enzimas[enzimas["gene_name"].isin(["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"])]
#Las que se metabolizan por mas de uno manteniendo la primera
duplicados = top_5[top_5.duplicated(subset="drugbank_id", keep=False)]
#Dejamos un unico id para que quede constancia de cuantos son
final = duplicados["drug_name"].unique().tolist()

len(final)